In [ ]:

from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict, Annotated
import operator
import os

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

# render markdown content in Notebook
from IPython.display import display, Markdown  #x mostrare output “belli” formattati nel notebook, display() x render anche imgs ect, Markdown() x render markdown
from scripts import utils  #my files

LLM_MODEL = "qwen3"
BASE_URL = "http://localhost:11434"

llm = ChatOllama(
        model=LLM_MODEL,
        base_url=BASE_URL
    )

utils.extract_filters("what is amazon's cashflow in 2024")

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

In [ ]:
@tool
def retrieve_docs(query:str, k=5):
    """
    Retrieve relevant financial documents from ChromaDB.
    Extracts filters from query and retrieves matching documents.
    Args:
        query: The search query (e.g., "What was Amazon's revenue in Q2 2025?")
        k: Number of documents to retrieve. generally prefer 5 docs
    Returns:
        Retrieved documents with metadata as formatted string
    """
    print(f"\n[TOOL] retrieve_docs called")
    print(f"[QUERY] {query}")
    filters = utils.extract_filters(query)
    ranking_keywords = utils.generate_ranking_keywords(query)
    # fetch more docs than needed for better re-ranking
    results = utils.search_docs(query, filters, ranking_keywords, k=10*k)
    # rank retrieved docs
    docs = utils.rank_documents_by_keywords(results, ranking_keywords, k=k)
    print(f"[RETRIEVED] {len(docs)} documents")
    # format extracted docs or chunks
    if len(docs)==0:
        return f"No ducuments found for the query: '{query}'. Try rephrasing query or use different filter."
    # final format
    # --- Document {i} ---
    retrieved_text = []
    for i, doc in enumerate(docs, 1):
        doc_text = [f"--- Document {i} ---"]
        # add all metadata
        for key, value in doc.metadata.items():
            doc_text.append(f"{key}: {value}")
        # add content
        doc_text.append(f"\nContent:\n{doc.page_content}")
        text = "\n".join(doc_text)
        retrieved_text.append(text)
    retrieved_text = "\n".join(retrieved_text)
    os.makedirs("debug_logs", exist_ok=True)
    with open("debug_logs/retrieved_reranked_docs.md", "w", encoding='utf-8') as f:
        f.write(retrieved_text)
    return retrieved_text

query = "what is amazon's revenue in 2024?"
result = retrieve_docs.invoke(query)

In [ ]:
def agent_node(state: AgentState):
    messages = state['messages']
    tools = [retrieve_docs]
    llm_with_tools = llm.bind_tools(tools)
    system_prompt = """You are a financial document analysis assistant with access to a document retrieval tool.
        CRITICAL RULES:
        1. ALWAYS use the retrieve_docs tool first - NEVER answer from memory
        2. You MUST call the tool before providing any financial information
        3. Answer ONLY based on the retrieved documents
        4. If documents don't contain the answer, clearly state that
        WORKFLOW FOR SIMPLE QUESTIONS:
        Step 1: Call retrieve_docs tool with the user's question
        Step 2: Wait for the tool results
        Step 3: Analyze the retrieved documents
        Step 4: Provide answer with citations (company, year, quarter, page)
        WORKFLOW FOR COMPLEX/COMPARISON QUESTIONS:
        Step 1: Break down the question into sub-questions
        Example: "Compare Amazon and Google revenue" -> 
        - Sub-question 1: "Amazon revenue"
        - Sub-question 2: "Google revenue"
        Step 2: Call retrieve_docs for EACH sub-question separately
        - First call for Amazon
        - Wait for results
        - Second call for Google
        - Wait for results
        Step 3: Analyze all retrieved documents
        Step 4: Present comparison in TABLE format:
        | Metric | Company A | Company B |
        |--------|-----------|-----------|
        | Revenue | $X | $Y |
        ANSWER FORMATTING (Use Markdown):
        - Use **headings** (##, ###) for sections
        - Use paragraphs for detailed findings and reasonings
        - Use **bullet points** for lists
        - Use **tables** for comparisons and structured data
        - Use **bold** for emphasis on key metrics
        - Cite sources: (Company: X, Year: Y, Quarter: Z, Page: N)
        EXAMPLES:
        Example 1 - Simple Question:
        User: "What was Amazon's revenue in Q2 2025?"
        You: [Call tool] -> [Analyze docs] -> 
        "## Amazon Q2 2025 Revenue
        Amazon's revenue for Q2 2025 was **$XXX billion**
        **Source:** Amazon, 2025, Q2, Page 5"
        Example 2 - Comparison Question:
        User: "Compare Amazon and Google revenue"
        You: [Call tool for Amazon] -> [Call tool for Google] -> [Analyze both] ->
        "## Revenue Comparison
        | Company | Revenue | Year | Quarter |
        |---------|---------|------|---------|
        | Amazon  | $XXX B  | 2025 | Q2      |
        | Google  | $YYY B  | 2025 | Q2      |
        **Analysis:**
        - Amazon's revenue was higher by $ZZZ billion
        - Google showed XX% growth
        **Sources:**
        - Amazon: 2025, Q2, Page 5
        - Google: 2025, Q2, Page 8"
        Example 3 - Multi-part Question:
        User: "What are Amazon's revenue, profit, and cash flow?"
        You: [Call tool] -> [Analyze docs] ->
        "## Amazon Financial Metrics
        ### Revenue
        - Q2 2025: $XXX billion
        ### Profit  
        - Q2 2025: $YYY billion
        ### Cash Flow
        - Q2 2025: $ZZZ billion
        **Source:** Amazon, 2025, Q2, Pages 5-7"
        REMEMBER:
        - ALWAYS call the tool first
        - Break complex questions into sub-questions
        - Use tables for comparisons
        - Format answers in detailed Markdown
        - Always cite sources
        - If no relevant documents are found, try with different filters."""
    system_msg = SystemMessage(system_prompt)
    messages = [system_msg] + messages
    response = llm_with_tools.invoke(messages)
    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tc in response.tool_calls:
            print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
    else:
        print(f"[AGENT] Responding...")
    return {'messages': [response]}

In [ ]:
def should_continue(state: AgentState):
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return "tools"
    else:
        return END

In [ ]:

def create_agent():
    builder = StateGraph(AgentState)
    builder.add_node('agent', agent_node)
    builder.add_node('tools', ToolNode([retrieve_docs]))
    # create edges
    builder.add_edge(START, 'agent')
    builder.add_conditional_edges('agent', should_continue, ['tools', END])
    builder.add_edge('tools', 'agent')
    graph = builder.compile()
    return graph

agent = create_agent()
agent

In [ ]:

query = "what is the amazon's revenue in 2023?"
result = agent.invoke({'messages': [HumanMessage(query)]})
result

result['messages'][-1].pretty_print()
display(Markdown(result['messages'][-1].content))

query = "what is the revenue of amazon's and google in 2023?"
result = agent.invoke({'messages': [HumanMessage(query)]})
display(Markdown(result['messages'][-1].content))